In [1]:
!pip install matplotlib

In [2]:
import json
import os
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np

In [3]:
if not os.path.exists("data"):
    os.makedirs("data")
print ("Data folder ready.")

Data folder ready.


In [4]:
def lbs_to_kg(pounds):
    return pounds/2.2
def kg_to_lbs(kg):
    return kg*2.2
def inches_to_meters(inches):
    return inches *0.0254
def meters_to_inches(meters):
    return meters/0.0254
def meters_to_centimeters(meters):
    centimeters =  meters*100
    return centimeters
def centimeters_to_meters(centimeters):
    meters = centimeters / 100
    return meters
def calculate_bmi_imperial(weight_lbs, height_inches):
    bmi =  (703 * weight_lbs) / (height_inches ** 2)
    return bmi
def calculate_bmi_metric(weight_kg,height_meters):
    bmi = weight_kg/(height_meters ** 2)
    return bmi


def get_bmi_category(bmi):
    if bmi < 18.5:
        return "Underweight"
    elif 18.5 <= bmi < 24.9:
        return "Normal weight"
    elif 25 <= bmi < 29.9:
        return "Overweight"
    else:
        return "Obesity"

def calculate_bmr(weight_kg,height_centimeters, gender, age):
    if gender == "male":
        bmr = 88.36 + (13.4*weight_kg) + (4.8*height_centimeters) - (5.7*age)
    else:
        bmr = 447.6 + (9.25*weight_kg) + (3.1*height_centimeters) - (4.33*age)
    return bmr

def calculate_tdee(bmr,activity_level):
    '''
    calculate tdee(total daily energy expenditure)
    '''
    activity_multiplier = {
        "low" : 1.4,
        "medium" : 1.55,
        "high" : 1.7,
    }
    multiplier = activity_multiplier[activity_level]
    tdee = bmr * multiplier
    return tdee

def calculate_daily_target(target_weight_kg, current_weight_kg, tdee, days):
    total_weight_difference = target_weight_kg - current_weight_kg
    total_calories_difference = total_weight_difference * 7700 * 0.7
    # 1kg fat approximately equal to 7700 kcal
    daily_calories_difference = total_calories_difference / days
    daily_target = tdee + daily_calories_difference
    return daily_target

In [5]:
def create_new_user(username):
    print(f"\nWelcome, {username}, let's set up.")
    age= int(input("\nEnter your age:"))
    gender = input("\nEnter your gender:\n**Currently we only support 'male' or 'female' based on available data ")
    
    print("\nChoose your preferred unit system:")
    print("1. Imperial (pounds, inches)")
    print("2. Metric (kg, meters)")
    unit_choice = input("Enter 1 or 2:")
    if unit_choice == "1":
        unit_system = "imperial"
        weight_lbs = float(input("\nEnter your weight (lbs):"))
        height_inches = float(input("\nEnter your height (inches):"))
        
        weight_kg = lbs_to_kg(weight_lbs)
        height_meters = inches_to_meters(height_inches)

    else:
        unit_system = "metric"
        weight_kg = float(input("\nEnter your weight (kgs):"))
        height_meters = float(input("Enter your height (meters): "))
        
        weight_lbs = kg_to_lbs(weight_kg)
        height_inches = meters_to_inches(height_meters)


    print("\nChoose your activity level:")
    print("Low: Sedentary (little or no exercise)")
    print("Medium: Moderately active (exercise 3-5 days/week)")
    print("High: Very active (exercise 6-7 days/week)")
    activity_level = input("Enter your activity level (low/medium/high): ").lower()

    profile = {
        "username" : username,
        "age" : age,
        "gender" : gender,
        "weight_kgs" : weight_kg,
        "weight_lbs": weight_lbs,
        "height_meters": height_meters,
        "height_inches": height_inches,
        "activity_level": activity_level,
        "unit_system": unit_system
    }

    has_target = input("\nDo you have a target weight goal? (Yes/No)").lower()
    if has_target == "yes":
        if unit_system == "imperial":
            target_lbs = float(input("Enter your target weight (lbs): "))
            profile['target_weight_kg'] = lbs_to_kg(target_lbs)
            profile['target_weight_lbs'] = target_lbs
        else:
            target_kg = float(input("Enter your target weight (kg): "))
            profile['target_weight_kg'] = target_kg
            profile['target_weight_lbs'] = kg_to_lbs(target_kg)
        
        days = int(input("Enter your target days to achieve your goal:"))
        profile["target_days"] = days
    else:
        print("\nNo problem, you can set your goal later")

    save_user_profile(username, profile)
    user_file = f"data/user_{username.lower()}.json"
    print (f"\nYour profile is created in {user_file}")
    return profile

def update_user_profile(username, existing_profile):
    print(f"\nHi {username}, update your information")
    print("\nWhat would you like to update?")
    print("1. Weight")
    print("2. Height")
    print("3. Activity level")
    print("4. Target weight goal")
    print("5. Unit system")
    print("6. All your information ")
    print("7. Cancel")

    choice = input("\nEnter your choice:(1-7)")

    if choice == "1":
        if existing_profile['unit_system'] == 'imperial':
            new_weight_lbs = float(input("Enter new weight (lbs): "))
            existing_profile['weight_lbs'] = new_weight_lbs
            existing_profile['weight_kg'] = lbs_to_kg(new_weight_lbs)
        else:
            new_weight_kg = float(input("Enter new weight (kg): "))
            existing_profile['weight_kg'] = new_weight_kg
            existing_profile['weight_lbs'] = kg_to_lbs(new_weight_kg)
        print("\nWeight updated!")
    elif choice =="2":
        if existing_profile['unit_system'] == 'imperial':
            new_height_inches = float(input("Enter new height (inches): "))
            existing_profile['height_inches'] = new_height_inches
            existing_profile['height_meters'] = inches_to_meters(new_height_inches)
        else:
            new_height_meters = float(input("Enter new height (meters): "))
            existing_profile['height_meters'] = new_height_meters
            existing_profile['height_inches'] = meters_to_inches(new_height_meters)
        print("\nHeight updated!")
    elif choice =="3":
        print("Activity Level:")
        print("Low")
        print("Medium")
        print("High")
        new_activity = input("Enter new activity level: ").lower()
        existing_profile['activity_level'] = new_activity
        print("\nActivity level updated!")
    elif choice == "4":
        if existing_profile['unit_system'] == 'imperial':
            target_lbs = float(input("Enter target weight (lbs): "))
            existing_profile['target_weight_kg'] = lbs_to_kg(target_lbs)
            existing_profile['target_weight_lbs'] = target_lbs
        else:
            target_kg = float(input("Enter target weight (kg): "))
            existing_profile['target_weight_kg'] = target_kg
            existing_profile['target_weight_lbs'] = kg_to_lbs(target_kg)
        
        days = int(input("In how many days? "))
        existing_profile['target_days'] = days
        print("\nTarget weight goal updated!")
    elif choice == "5":
        print(f"Current unit system: {existing_profile['unit_system']}")
        new_system = input("Switch to (imperial/metric): ").lower()
        
        if new_system in ['imperial', 'metric']:
            existing_profile['unit_system'] = new_system
            print(f"Unit system changed to {new_system}!")
        else:
            print("\nInvalid unit system. No changes made.")
    elif choice == "6":
        print("\nRe-creating your entire profile")
        return create_new_user(username)
    elif choice == "7":
        print("\nUpdate canceled")
        return existing_profile

    save_user_profile(username, existing_profile)
    return existing_profile
    
    
    
        
    
    
def get_all_users():
    '''
    get and return list of usernames
    '''
    try:
        with open("data/users.json","r")as f:
                  return json.load(f)
            # transform json file into python data type that we can use in program
    except FileNotFoundError:
        return[]
def user_exists(username):
    '''
    this transform usernames into lower case and returns a bool whether the user exists
    '''
    users = get_all_users()
    return username.lower() in [u.lower() for u in users]
def save_user_profile(username, profile_data):
    filename = f'data/user_{username.lower()}.json'
    #create a file linked to the username#
    profile_data['last_updated'] = datetime.now().strftime('%m-%d-%Y %H-%M-%S')

    with open(filename, 'w') as f:
        json.dump(profile_data,f,indent = 2)
    # save all user information in json file

    users = get_all_users()
    if username not in users:
        users.append(username)
        with open ('data/users.json','w')as f:
            json.dump(users, f)
            #this is used to use a json file
    print(f'\nUser profile is saved for {username}')
def load_user_profile(username):
    filename = f'data/user_{username.lower()}.json'
    try:
        with open(filename,'r')as f:
            return json.load(f)
    except FileNotFoundError:
        return None
def display_profile(profile):
    print("\nYour Information:")
    for key,value in profile.items():
        print(f"{key},{value}")

In [6]:
#This cell might be marked as got help from chatbots because I had trouble with database processing 
def create_simple_food_database():
    
    print("=" * 60)
    print("Processing FoodData Central Database...")
    print("=" * 60)
    
    print("\n[1/5] Loading food.csv...")
    foods = pd.read_csv("FoodData_Central/food.csv", low_memory=False)
    print(f"✓ Loaded {len(foods):,} foods")

    
    print("\n[2/5] Loading food_nutrient.csv...")
    nutrients_data = pd.read_csv("FoodData_Central/food_nutrient.csv", low_memory=False)
    print(f"✓ Loaded {len(nutrients_data):,} nutrient records")

    
    print("\n[3/5] Filtering nutrients...")
    needed_nutrients = {
        1008: 'Calories',
        1003: 'Protein',
        1005: 'Carbs',
        1004: 'Fat'
    }
    
    nutrients_filtered = nutrients_data[
        nutrients_data['nutrient_id'].isin(needed_nutrients.keys())
    ]
    print(f"✓ Filtered to {len(nutrients_filtered):,} relevant records")

    
    print("\n[4/5] Reshaping data...")
    nutrients_pivot = nutrients_filtered.pivot_table(
        index='fdc_id',
        columns='nutrient_id',
        values='amount',
        aggfunc='first'
    )
    
    nutrients_pivot.columns = [needed_nutrients.get(col, col) for col in nutrients_pivot.columns]
    nutrients_pivot = nutrients_pivot.reset_index()

    
    print("\n[5/5] Merging data...")
    final_db = foods[['fdc_id', 'description']].merge(
        nutrients_pivot,
        on='fdc_id',
        how='inner'
    )
    
    final_db = final_db.rename(columns={'description': 'Food'})
    final_db = final_db.dropna(subset=['Calories'])


    output_path = "data/food_database_simple.csv"
    final_db.to_csv(output_path, index=False)
    
    print("\n" + "=" * 60)
    print(f"✓ SUCCESS! Created database with {len(final_db):,} foods")
    print(f"✓ Saved to: {output_path}")
    print("=" * 60)
    
    return final_db

food_db = create_simple_food_database()
food_db.head(10)

Processing FoodData Central Database...

[1/5] Loading food.csv...
✓ Loaded 74,175 foods

[2/5] Loading food_nutrient.csv...
✓ Loaded 155,243 nutrient records

[3/5] Filtering nutrients...
✓ Filtered to 6,422 relevant records

[4/5] Reshaping data...

[5/5] Merging data...

✓ SUCCESS! Created database with 135 foods
✓ Saved to: data/food_database_simple.csv


,fdc_id,Food,Protein,Fat,Carbs,Calories
53,321358,"Hummus, commercial",7.35,17.10,14.90,229.0
54,321359,"Milk, reduced fat, fluid, 2% milkfat, with add...",3.35,1.90,4.91,50.0
55,321360,"Tomatoes, grape, raw",0.83,0.63,5.51,27.0
68,321611,"Beans, snap, green, canned, regular pack, drai...",1.04,0.39,4.11,21.0
87,321900,"Broccoli, raw",2.57,0.34,6.29,32.0
114,322228,"Milk, lowfat, fluid, 1% milkfat, with added vi...",3.38,0.95,5.19,43.0
142,322559,"Milk, nonfat, fluid, with added vitamin A and ...",3.43,0.08,4.89,34.0
169,322892,"Milk, whole, 3.25% milkfat, with added vitamin D",3.28,3.20,4.67,60.0
187,323121,"Frankfurter, beef, unheated",11.70,28.00,2.89,314.0
201,323294,"Nuts, almonds, dry roasted, with salt added",20.40,57.80,16.20,620.0


In [7]:
'''
def load_food_database(csv_path = "data/food_database_simple.csv"):
    try:
        df = pd.read_csv(csv_path)
        df["Calories"] = df["Calories"].astype(str).str.extract(r'(\d+\.?\d*)')
        # df["Calories"].astype(str) converts the “Calories” column values to strings
        # (This ensures you can apply text/regex operations even if the column has numbers or NaN.)
        # .str.extract(r'(\d+\.?\d*)') Uses a regular expression (regex) to pull out numeric parts from each string, \d+ → one or more digits, 
        # \.? → optional decimal point, \d* → zero or more digits after the decimal
        # after this line, the “Calories” column contains only the numeric part as text (not yet numbers).
        df["Calories"] = pd.to_numeric(df["Calories"], errors="coerce")
        # pd.to_numeric() converts the extracted text values into real numeric (float) values
        # errors="coerce" means: If a value can’t be converted to a number (e.g., NaN, "unknown", etc.), it turns into NaN instead of raising an error.
        print("\nFood Database loaded")
        return df
    except FileNotFoundError:
        print(f"\nSorry! File not found! {csv_path}")
'''


def load_food_database(csv_path = "data/food_database_simple.csv"):
    try:
        df = pd.read_csv(csv_path)
        for col in ["Calories","Protein","Carbs","Fat"]:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col],errors = "coerce")

        df = df.dropna(subset = ["Calories"])

        print(f"Database loaded, {len(df)} foods available")

        return df

    except FileNotFoundError:
        print(f"\nSorry! File not found! {csv_path}")
        
        return None

food_db = load_food_database()



Database loaded, 135 foods available


In [8]:

def load_daily_intake (username, date = None):
    if date == None:
        date = datetime.now().strftime("%Y-%m-%d")

    filename = f"data/intake_{username.lower()}.json"

    try:
        with open (filename, "r")as f:
            all_data = json.load(f)
    except FileNotFoundError:
        all_data = {
            'username':username,
            'history':{}
        }

    if date in all_data['history']:
        return all_data['history'][date]
    else:
        return{
            'date':date,
            'foods':[],
            'total_calories':0,
            'total_protein':0,
            'total_carbs':0,
            'total_fat':0
        
        }
        

def save_daily_intake(username, intake_data):
   
    filename = f"data/intake_{username.lower()}.json"

    try:
        with open(filename,'r')as f:
            all_data = json.load(f)
    except FileNotFoundError:
        all_data={
            'username':username,
            'history':{}
        }

    date = intake_data['date']
    all_data['history'][date] = intake_data

    with open (filename, 'w') as f:
        json.dump(all_data, f ,indent = 2)

    print("Daily intake saved")

def add_food_manual(intake):
    print("\n Manual Food Entry:")
    food_name = input("Food name:").strip()
    calories = float(input("Calories:"))

    has_nutrition = input("Do you know all the protein/carbs/fat details? (yes/no):").lower()

    if has_nutrition =="yes":
        protein = float(input("Protein(g):"))
        carbs = float(input("Carbs(g):"))
        fat = float(input("Fat(g):"))
    
    
        food_entry = {
            "name" : food_name,
            "calories": calories,
            "protein": protein,
            "carbs": carbs,
            "fat": fat,
            "source": "manual"
        }
    
        intake["foods"].append(food_entry)
        intake["total_calories"] += calories
        intake["total_protein"] += protein
        intake["total_carbs"] += carbs
        intake["total_fat"] += fat
    
        print(f"\nAdded: {food_name} : {calories} kcal")
    else:
        print("Sorry, manual input requires entering all the details")
        
    return intake
    
def add_food_database(intake, food_db):
    search_term = input("Search for your intake food:").strip().lower()

    results = food_db[food_db["Food"].str.lower().str.contains(search_term)]

    if len(results) ==0:
        print("Sorry, no food found")
        return intake

    print(f"Found {len(results)} foods:")
    print (results[["fdc_id","Food","Calories"]].to_string(index = False))

    select_id = int(input("\n Food ID:"))
    select_food = results[results["fdc_id"] == select_id]

    if len(select_food) == 0:
        print("\nInvalid Id! Please enter again!")
        return intake
    
    food = select_food.iloc[0]
    
    while True:
        try:
            serving = float(input("Serving:(1 serving = 100g)"))
            break
        except ValueError:
            print("Please enter a valid number")

    food_entry = {
        "name": food["Food"],
        "calories": food["Calories"] * serving,
        "protein": food["Protein"] * serving,
        "carbs": food["Carbs"] * serving,
        "fat": food["Fat"] * serving,
        "source": "database"
    }
    intake["foods"].append(food_entry)
    intake["total_calories"] += food_entry["calories"]
    intake["total_protein"] += food_entry["protein"]
    intake["total_carbs"] += food_entry["carbs"]
    intake["total_fat"] += food_entry["fat"]

    print(f"\nAdded: {food_entry["name"]} : {food_entry["calories"]} kcal")

    return intake


def display_daily_intake(intake,profile):
    print("Today's intake:")

    if len(intake["foods"]) == 0:
        print("\nEat and record something~")
    else:
        for food in intake["foods"]:
            print(f"\n{food['name']} - {food['calories']:.1f} kcal")
            
        print(f"\nTotal Calories: {intake['total_calories']:.1f} kcal")
        print(f"\nProtein: {intake['total_protein']:.1f} g")
        print(f"\nCarbs: {intake['total_carbs']:.1f} g")
        print(f"\nFat: {intake['total_fat']:.1f} g")

#Chatbots
def get_food_suggestions(intake, profile, food_db):

    height_cm = meters_to_centimeters(profile['height_meters'])
    bmr = calculate_bmr(profile['weight_kgs'], height_cm, profile['gender'], profile['age'])
    tdee = calculate_tdee(bmr, profile['activity_level'])
    
    if 'target_weight_kg' in profile:
        daily_target_calories = calculate_daily_target(
            profile['target_weight_kg'],
            profile['weight_kgs'],
            tdee,
            profile['target_days']
        )
    else:
        daily_target_calories = tdee
    
 
    remaining_calories = daily_target_calories - intake['total_calories']
    protein_target = profile['weight_kgs'] * 1.8
    remaining_protein = protein_target - intake['total_protein']

    
    print(f"\n{'='*60}")
    print("Food Recommendations")
    print(f"{'='*60}")
    print(f"\nYou still need:")
    print(f"  Calories: {remaining_calories:.0f} kcal")
    print(f"  Protein: {remaining_protein:.0f}g")
    
    if remaining_calories <= 0 and remaining_protein <= 0:
        print("\n✓ Goals met!")
        return
    

    if remaining_protein > 10:
        print(f"\n{'='*60}")
        print("Suggested high-protein foods:")
        print(f"{'='*60}")
        

        common_protein_foods = [
            'chicken',
            'turkey',
            'beef',
            'fish',
            'egg, whole',
            'cheese',
            'yogurt'
        ]
        
        recommendations = []
        
        for keyword in common_protein_foods:
            matches = food_db[
                food_db['Food'].str.lower().str.contains(keyword, na=False) &
                (food_db['Protein'] > 10) &  
                (food_db['Protein'] < 50) &  
                (food_db['Calories'] <= remaining_calories)
            ]
            
            if len(matches) > 0:
                best = matches.sort_values('Protein', ascending=False).iloc[0]
                recommendations.append(best)
        
        if len(recommendations) > 0:
            rec_df = pd.DataFrame(recommendations).drop_duplicates(subset=['fdc_id'])
            rec_df = rec_df.sort_values('Protein', ascending=False).head(5)
            
            for _, food in rec_df.iterrows():
                print(f"  • {food['Food']}")
                print(f"    {food['Calories']:.0f} kcal | {food['Protein']:.1f}g protein")
        else:
            print("  No suitable foods found")
    
 
    elif remaining_calories > 100:
        print(f"\n{'='*60}")
        print("Balanced food suggestions:")
        print(f"{'='*60}")
        
        balanced = food_db[
            (food_db['Calories'] >= 100) &
            (food_db['Calories'] <= min(remaining_calories, 500))
        ].head(5)
        
        for _, food in balanced.iterrows():
            print(f"  • {food['Food']} - {food['Calories']:.0f} kcal")
    
    print(f"\n{'='*60}")

In [9]:
def log_food_intake(profile, food_db):
    username = profile['username']
    intake = load_daily_intake(username)

    print(f"\nLogging food intake - {intake['date']}")

    while True:
        print("\n1. Manual entry(requires entering all the details)")
        print("2. Search database to add foods")
        print("3. View today's intake")
        print("4. Get food suggestions")
        print("5. Return")

        choice = input ("\nChoice:(1-5) ").strip()

        if choice == "1":
            intake = add_food_manual(intake)
            save_daily_intake(username, intake)
        elif choice == "2":
            intake = add_food_database(intake, food_db)
            save_daily_intake(username, intake)
        elif choice == "3":
            display_daily_intake(intake, profile)
        elif choice == "4":
            get_food_suggestions(intake,profile,food_db)
        elif choice == "5":
            save_daily_intake(username, intake)
            print("\nChanges saved")
            break
        else:
            print("Please enter a valid number")

In [10]:
def view_history(username,profile):
    filename = f'data/intake_{username.lower()}.json'
    try:
        with open (filename,'r')as f:
            all_data = json.load(f)
    except FileNotFoundError:
        print("No history found. Please log some food first~")
        return

    history = all_data['history']

    if len(history) == 0:
        print("No history found. Please log some food first~")
        return

    dates_list = sorted(history.keys())
    dates = [datetime.strptime(d, "%Y-%m-%d").date() for d in dates_list]

    
    daily_kcal = []
    for date_str in dates_list:
        day_data = history[date_str]
        # total_calories(Diwen type) ➡️ total_calories(Abby type)
        daily_kcal.append({
            "date" :datetime.strptime(date_str, "%Y-%m-%d").date(),
            "total_kcal" : day_data.get('total_calories', 0 ),
            "foods" : day_data.get('foods' , [])
        })


    # User information
    user_info = {
        "Name": username,
        "Gender": "Male" if profile['gender'] == "male" else "Female",
        "Height (cm)": int(meters_to_centimeters(profile['height_meters'])),
        "Weight (kg)": profile['weight_kgs'],
        "Age": profile['age']
    }
    
    # Calculate user's BMR using the calculate_bmr function by Diwen
    bmr = calculate_bmr(user_info["Weight (kg)"], user_info["Height (cm)"], user_info["Gender"], user_info["Age"])
    
    
    # Convert to DataFrame
    daily_kcal_df = pd.DataFrame(daily_kcal)
    
    # Simulating additional exercise calories (random between 200~400 kcal)
    extra_exercise_kcal = np.random.randint(200, 400, size=len(dates))
    
    # Total expenditure calories = BMR + exercise calories
    total_expenditure_kcal = bmr + extra_exercise_kcal
    
    # Calculate calorie gap and adjust weight based on calorie intake
    initial_weight = user_info["Weight (kg)"]
    weights_adjusted = [initial_weight]
    
    for i in range(1, len(dates)):
        # Adjust weight based on calorie intake
        calorie_intake = daily_kcal_df.loc[i, "total_kcal"]
        weight_change = calorie_intake / 7700  # 7700 kcal = 1kg of weight loss
        new_weight = weights_adjusted[i-1] - weight_change
        weights_adjusted.append(round(new_weight, 1))  # Round to 1 decimal place
    
    # Plot weight change trend, calorie intake and expenditure
    fig, ax1 = plt.subplots(figsize=(10, 6))
    
    # Plot weight change (left axis, blue line)
    ax1.plot(dates, weights_adjusted, label="Weight Change (kg)", marker='o', color='b', linestyle='-', markersize=5)
    ax1.set_xlabel("Date")
    ax1.set_ylabel("Weight (kg)", color='b')
    ax1.tick_params(axis='y', labelcolor='b')
    
    # Create right axis, plot calorie intake and expenditure
    ax2 = ax1.twinx()
    ax2.plot(dates, daily_kcal_df['total_kcal'], label="Calorie Intake (kcal)", marker='o', color='orange', linestyle='-', markersize=5)
    ax2.plot(dates, total_expenditure_kcal, label="Calorie Expenditure (kcal)", marker='o', color='green', linestyle='-', markersize=5)
    ax2.set_ylabel("Calories (kcal)", color='g')
    ax2.tick_params(axis='y', labelcolor='g')
    
    # Add title and labels
    plt.title(f"{username}'s Weight Change and Calorie Intake/Expenditure")
    plt.xticks(rotation=45)
    
    # Display legend on the right side
    fig.tight_layout()
    fig.legend(loc="upper right", bbox_to_anchor=(1, 1), bbox_transform=ax1.transAxes)
    
    # Add the weight target line (if exists)
    if 'target_weight_kg' in profile:
        target_weight = profile['target_weight_kg']
        ax1.axhline(y=target_weight, color='r', linestyle='--', label=f'Target Weight ({target_weight} kg)')
        ax1.text(dates[0], target_weight + 0.2, "GOAL", color='r', fontsize=14, fontweight='bold')
    
    # Show the grid
    plt.grid(True)
    
    # Show plot
    plt.show()

In [11]:
def main():
    print("Welcome to Food Suggestions Giver!")
    username = input("\nEnter your name:")
    if user_exists(username):
        profile = load_user_profile(username)
        print(f"\nWelcome back, {username}")
        print("\nYour current information:")
        display_profile(profile)
        update_choice = input("\nWould you like to update your information? (yes/no)")
        if update_choice =="yes":
            profile = update_user_profile(username, profile)

    else:
        print("\nIt seems you are a new user, let's get set up!")
        profile = create_new_user(username)

    print("\nYour profile setup complete!")

    print("Loading food database...")
    food_db = load_food_database()

    while True:
        print(f"Hello, {username}!")
        print("\n1. View my profile")
        print("2. Update my profile")
        print("3. Enter/View today's intake")
        print("4. View history & visualization")
        print("5. Exit")
        choice = input("\nChoice (1-5): ").strip()
        
        if choice == "1":
            display_profile(profile)
        elif choice == "2":
            profile = update_user_profile(username, profile)
        elif choice == "3":
            log_food_intake(profile, food_db)
        elif choice == "4":
            view_history(username, profile)
        elif choice == "5":
            print(f"\nGoodbye, {username}! ")
            break
        else:
            print("Please enter a valid number")

In [ ]:
main()

Welcome to Food Suggestions Giver!


In [ ]:
print(os.getcwd())